# 04 — Error Analysis

Analyze where the model fails by examining misclassified samples.

In [18]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import sys


# Find project root (one level above notebook folder)
ROOT = Path().resolve().parents[0]   # adjust if needed
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))


RESULTS_DIR = Path("../outputs/results")
TEST_CSV = Path("../data/processed/test.csv")

metrics = json.load(open(RESULTS_DIR / "metrics.json"))
print("Metrics:", json.dumps(metrics, indent=2))

Metrics: {
  "accuracy": 0.8138208654813821,
  "macro_f1": 0.7997592014271312,
  "per_class_f1": {
    "positive": 0.8435434111943587,
    "negative": 0.8610987047789191,
    "neutral": 0.6946354883081155
  },
  "trainable_params": 394499
}


In [13]:
print(SRC)

/Users/saikrishnakowda/Documents/Projects/Tenglish-Sentiment/src


## Misclassified Samples

Reload test predictions (if saved) or load the confusion matrix.

In [ ]:
# Re-run predictions to get misclassified samples
import torch
from model import TenglishModel
from dataset import TenglishDataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from utils import load_checkpoint


if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")
model = TenglishModel().to(device)
ckpt = load_checkpoint(model, None, "../outputs/checkpoints/best_model_lora_only.pt")
model.eval()

test_ds = TenglishDataset(TEST_CSV)
test_loader = DataLoader(test_ds, batch_size=32)

y_true, y_pred = [], []
texts = []
@torch.no_grad()
def predict():
    for batch in test_loader:
        view1_ids = batch["view1_input_ids"].to(device)
        view1_mask = batch["view1_attention_mask"].to(device)
        view2_ids = batch["view2_input_ids"].to(device)
        view2_mask = batch["view2_attention_mask"].to(device)
        labels = batch["label"]
        
        logits = model(view1_ids, view1_mask, view2_ids, view2_mask)["logits"]
        preds = logits.argmax(1).cpu()
        y_true.extend(labels)
        y_pred.extend(preds)
predict()

errors = [i for i, (t, p) in enumerate(zip(y_true, y_pred)) if t != p]
print(f"Total errors: {len(errors)} / {len(y_true)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8497.67it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total errors: 1614 / 2981


In [24]:
error_df = test_ds.df.iloc[errors].copy()
error_df["predicted"] = [y_pred[i] for i in errors]

LABEL_NAMES = ["positive", "negative", "neutral"]

# Handle label types safely
if isinstance(error_df["label"].iloc[0], int):
    error_df["true_label"] = error_df["label"].map(lambda x: LABEL_NAMES[x])
else:
    error_df["true_label"] = error_df["label"]

error_df["pred_label"] = error_df["predicted"].map(lambda x: LABEL_NAMES[x])

print("Most common error types:")
print(
    error_df.groupby(["true_label", "pred_label"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print("\nSample misclassified sentences:")
print(
    error_df[["text_roman", "true_label", "pred_label"]]
    .head(10)
    .to_string()
)

Most common error types:
true_label  pred_label
negative    positive      995
positive    neutral       256
neutral     positive      201
negative    neutral       162
dtype: int64

Sample misclassified sentences:
                                                                                                                                                                                                                                                                                text_roman true_label pred_label
0                                                                                                                                                                                                                                                 Emotional ga connect avthadi second half   positive    neutral
2                                                                                                                                                                               